
## 입찰메이트 RAG — 작업 히스토리 v9

---

### 1. 파일 구조 파악 (초기 검토)

**업로드된 파일 전체 분석 완료:**

| 파일 | 역할 | 상태 |
|---|---|---|
| `retrieval_interface_F.py` | Retrieval 핵심 (BidMateRetriever, EMBED_CONFIG, build_prompt) | 수정됨 |
| `retrieval_eval_F.py` | 평가 전용 Retriever (BidMateEvaluator) | 수정됨 |
| `generation_interface.py` | LLM 생성 모듈 | 수정됨 |
| `serving_main.py` | Retrieval+Generation 조합, 히스토리 관리 | 수정됨 |
| `serving_main_1.py` | serving_main 개선버전 (C타입 판별 로직 개선) | → serving_main으로 교체 완료 |
| `fastapi_app.py` | FastAPI 엔드포인트 | 수정됨 |
| `run_eval_F.py` | 배치 평가 실행기 | 수정됨 |
| `eval_e2e_ctype.py` | C타입 E2E 평가 | 수정됨 |
| `eval_e2e_d_e_type.py` | D/E타입 E2E 평가 | 새로 작성 |
| `eval_e2e_a_b_type.py` | A/B타입 E2E 평가 | 새로 작성 |
| `eval_quant_judge.py` | 단일 Judge 정량 평가 | 수정됨 |
| `eval_quant_judge_dual.py` | 듀얼 Judge 정량 평가 | 수정됨 |
| `eval_qual_analyzer.py` | 정성 평가 리포트 | 수정됨 |

---

### 2. 발견된 버그 및 수정 내역

**🔴 즉시 오류 (수정 완료):**

| 파일 | 버그 | 수정 |
|---|---|---|
| `retrieval_eval_F.py` | `from retrieval_interface import` → 파일명 오류 | `retrieval_interface_F`로 수정 |
| `eval_quant_judge.py` | `import os` 누락, `from typing import Optional`이 `__main__` 블록 안 | 상단으로 이동 |
| `eval_qual_analyzer.py` | `merged.get(컬럼명, 5)` → DataFrame에 `.get()` 사용 오류 | `merged[컬럼명].fillna(6)` 로 수정 |
| `eval_e2e_ctype.py` | 마지막 줄에 `s` 단독 존재 | 삭제 |
| `eval_e2e_a_b_type.py` | `target`이 모듈 레벨에 있던 버그, `_mid_b`에 `de_` 접두사 오류 | 수정 |

**🟡 기능 오류 (수정 완료):**

| 파일 | 버그 | 수정 |
|---|---|---|
| `generation_interface.py` | `AutoModelForCausalLM` → Gemma-3는 멀티모달이라 잘못된 클래스 | `AutoModelForImageTextToText`로 수정 |
| `eval_quant_judge_dual.py` | `max_tokens` → gpt-5-mini 미지원 | `max_completion_tokens` 자동 분기 추가 |
| `fastapi_app.py` | `__mro__[0].__dict__.get('_MAX_HISTORY_MSGS')` 패턴 | `from serving_main import _MAX_HISTORY_MSGS`로 수정 |

---

### 3. 시나리오 구조 개편

**기존 (A-1/A-2/B):**
```
A-1: KURE-v1 임베딩 + Gemma 로컬
A-2: KoE5 임베딩 + LLaMA 로컬
B  : OpenAI API
```

**변경 후 (임베딩_LLM 12개 조합):**
```
임베딩 3개: KURE / KOE5 / SMALL
LLM    4개: GEMMA / QWEN / PHI / OPENAI

조합: KURE_GEMMA, KURE_QWEN, KURE_PHI, KURE_OPENAI
     KOE5_GEMMA, KOE5_QWEN, KOE5_PHI, KOE5_OPENAI
     SMALL_GEMMA, SMALL_QWEN, SMALL_PHI, SMALL_OPENAI
```

**EMBED_CONFIG 변경:**
```python
# 기존
{'A-1': KURE, 'A-2': KoE5, 'B': SMALL}

# 변경
{'KURE': ..., 'KOE5': ..., 'SMALL': ...}
```

**_PROMPT_TEMPLATES 변경:**
```python
# 기존
{'A-1': Gemma포맷, 'A-2': LLaMA포맷, 'B': OpenAI포맷}

# 변경
{'GEMMA': ..., 'QWEN': ..., 'PHI': ..., 'OPENAI': ...}
```

**get_generator() 변경:**
```python
# 기존
if scenario in ('A-1', 'A-2'): LocalHFGenerator
elif scenario == 'B': APIGenerator

# 변경
llm_key = scenario.split('_')[1]
if llm_key == 'GEMMA': LocalHFGenerator
elif llm_key in ('QWEN', 'PHI'): OpenRouterGenerator
elif llm_key == 'OPENAI': APIGenerator
```

---

### 4. OpenAI 모델 설정

- **팀 허용 모델:** `gpt-5-mini`, `gpt-5-nano`, `text-embedding-3-small`
- **`gpt-5-mini`** → `max_completion_tokens` 파라미터 사용 (max_tokens 미지원)
- `_B_USE_COMPLETION_TOKENS` 플래그로 자동 분기 처리
- **현재 `gpt-5.4-mini` → `gpt-5-mini`로 교체 필요** (미완료)

```bash
# 아직 안 한 것
sed -i "s|gpt-5.4-mini|gpt-5-mini|g" \
    /home/euijeong/2Team_Project/hej/generation_interface.py \
    /home/euijeong/2Team_Project/hej/eval_quant_judge_dual.py
```

---

### 5. E2E 평가 실행 결과

**`eval_e2e_ctype.py` 실행 (KURE_GEMMA vs SMALL_OPENAI, C타입 61개):**
- GEMMA 생성: 완료 (1시간 34분, bfloat16 로드)
- OPENAI 생성: 완료 (2분 14분)
- 저장: `eval_results/generation/e2e_ctype_comparison.csv`
- 중간파일: `e2e_mid_KURE_GEMMA.csv`, `e2e_mid_SMALL_OPENAI.csv`

**결과 샘플 비교:**
- GEMMA: 문서 내용 나열, 질문에 직접 답 안 함 → 품질 낮음
- OPENAI(gpt-5-mini): 핵심 답변 + 출처 명시 → 품질 높음

---

### 6. OpenRouter 설정

**발급 완료:** `sk-or-v1-f6e4...` (노출됨 → 재발급 필요)

**무료 모델 변경 이력:**
```
qwen/qwen3-14b:free      → 404 (없음)
qwen/qwen3-235b-a22b:free → 404 (없음)
qwen/qwen3.6-plus-preview:free → 404 (없음)
google/gemma-4-26b-a4b-it:free → ✅ 정상 작동
```

**현재 `_OR_MODEL_MAP`:**
```python
'OR-GEMMA': 'google/gemma-3-4b-it:free',
'OR-QWEN' : 'qwen/qwen3-235b-a22b:free',  # ← 404, 수정 필요
'OR-PHI'  : 'microsoft/phi-4-mini-instruct:free',  # 미확인
'QWEN'    : 'qwen/qwen3-235b-a22b:free',  # ← 404, 수정 필요
'PHI'     : 'microsoft/phi-4-mini-instruct:free',  # 미확인
```

---

### 7. GCP 환경 이슈 및 해결

| 이슈 | 원인 | 해결 |
|---|---|---|
| `GatedRepoError` | HF 로그인 안 됨 | `hf auth login` 으로 토큰 입력 |
| `accelerate` 없음 | 패키지 미설치 | `pip install accelerate` |
| `bitsandbytes` 오류 | `libnvJitLink.so.13` 경로 없음 | `export LD_LIBRARY_PATH=.../nvidia/cu13/lib` |
| Gemma 로드 오류 | `AutoModelForCausalLM` → 멀티모달 모델에 맞지 않는 클래스 | `AutoModelForImageTextToText`로 변경 |
| `_QUANTIZE=True` 오류 | bitsandbytes CUDA 버전 불일치 | `_QUANTIZE=False`로 변경 (bfloat16 로드) |

---

### 8. 현재 남은 작업

**즉시 해야 할 것:**
1. ⚠️ 노출된 API 키 전부 재발급 (OpenAI 팀키 4회, OpenRouter 3회)
2. `gpt-5.4-mini` → `gpt-5-mini` 교체 (sed 명령어 미실행)
3. OpenRouter QWEN 모델 ID 수정 (`gemma-4-26b-a4b-it:free` 또는 다른 무료 모델)
4. PHI 모델 작동 여부 확인

**다음 실행 순서:**
1. `eval_e2e_d_e_type.py` — D/E타입 130개 E2E
2. `eval_e2e_a_b_type.py` — A/B타입 386개 E2E (선택)
3. `eval_quant_judge_dual.py` — Judge 채점
4. `eval_qual_analyzer.py` — 정성 평가

**LLM 시나리오 미확정:**
- A-1: Gemma 3-4b (GCP 로컬) — 파인튜닝 후보
- A-2: Qwen (VRAM 문제로 4B급만 가능) 또는 OpenRouter 대체
- A-3: Phi-4-mini (GCP 로컬 가능, ~8GB)
- B: gpt-5-mini (팀 OpenAI API)

**파인튜닝 여부:**
- 미션 필수 아님
- 결과 보고 결정 예정
- 학습 데이터: eval 579개 활용 (leakage 감수) 또는 청크에서 규칙 기반 생성

---

### 9. 파일별 최종 수정사항 요약

**`eval_e2e_ctype.py`:**
- 저장 경로: `eval_results/generation/`으로 분리
- 중간저장 + 재시작 시 중간파일 재사용 로직 추가
- SCENARIO_A/B 환경변수 기반, 컬럼명 동적 생성

**`eval_qual_analyzer.py`:**
- `_COL_A/B` 환경변수 기반 동적 처리
- `SCORE_PREFIX` 환경변수로 단일/듀얼 judge 컬럼명 자동 분기
- `import os` 추가

**`eval_quant_judge_dual.py`:**
- `_JUDGE_1_USE_COMPLETION_TOKENS` 플래그 추가
- 출력 컬럼명 `gemma_*/gpt_*` → `a_*/b_*` (시나리오 독립적)
- 환경변수 기반 컬럼명 동적 처리